# F1 Leakage-Safe Prediction Baselines

Two reproducible baselines: finishing position and pit-stop count. Features are computed from prior races only, and the latest season is held out chronologically. Scores are reference points—not claims of production readiness.


In [ ]:
import os
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path(os.getenv("F1_DATA_DIR", "/kaggle/input/formula-1-pit-stop-dataset"))
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_DIR}")
print(f"Reading data from {DATA_DIR}")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

drivers = pd.read_csv(DATA_DIR / "race_drivers.csv")
pits = pd.read_csv(DATA_DIR / "pit_events.csv")
context = pd.read_csv(DATA_DIR / "race_context.csv")
keys = ["season", "round_number", "driver_id"]

frame = drivers.merge(context[["season", "round_number", "circuit_id"]], on=["season", "round_number"], how="left")
stop_counts = pits.groupby(keys).size().rename("pit_stop_count").reset_index()
frame = frame.merge(stop_counts, on=keys, how="left")
# Zero is valid only inside the recorded-pit era (2011 onward).
frame.loc[frame["season"].ge(2011), "pit_stop_count"] = frame.loc[frame["season"].ge(2011), "pit_stop_count"].fillna(0)
frame = frame.sort_values(["season", "round_number", "driver_id"]).reset_index(drop=True)

# Collapse to one entity/race value before shifting. This prevents another
# entry in the same race from entering the current row's history.
def add_prior_mean(data, entity, value, output):
    race_value = (data.groupby([entity, "season", "round_number"], as_index=False)[value]
                  .mean().sort_values([entity, "season", "round_number"]))
    race_value[output] = race_value.groupby(entity)[value].transform(
        lambda s: s.shift().expanding().mean()
    )
    return data.merge(race_value[[entity, "season", "round_number", output]],
                      on=[entity, "season", "round_number"], how="left")

frame = add_prior_mean(frame, "driver_id", "classified_position", "driver_prior_mean_finish")
frame = add_prior_mean(frame, "constructor_id", "classified_position", "constructor_prior_mean_finish")
frame = add_prior_mean(frame, "driver_id", "pit_stop_count", "driver_prior_mean_stops")
frame.tail()


## Chronological evaluation helper


In [ ]:
categorical = ["driver_id", "constructor_id", "circuit_id"]
numeric = ["grid_position", "driver_prior_mean_finish", "constructor_prior_mean_finish", "driver_prior_mean_stops"]
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical),
])

def evaluate(data, target):
    data = data.dropna(subset=[target]).copy()
    test_season = int(data["season"].max())
    train, test = data[data["season"] < test_season], data[data["season"] == test_season]
    assert train["season"].max() < test["season"].min()
    X_train, X_test = train[numeric + categorical], test[numeric + categorical]
    y_train, y_test = train[target], test[target]
    models = {
        "median_dummy": Pipeline([("prep", prep), ("model", DummyRegressor(strategy="median"))]),
        "random_forest": Pipeline([("prep", prep), ("model", RandomForestRegressor(
            n_estimators=40, max_depth=12, min_samples_leaf=4, random_state=42, n_jobs=1
        ))]),
    }
    rows = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({"target": target, "model": name, "test_season": test_season,
                     "train_rows": len(train), "test_rows": len(test),
                     "MAE": mean_absolute_error(y_test, pred),
                     "RMSE": mean_squared_error(y_test, pred) ** .5})
    return pd.DataFrame(rows), test_season


## Finishing-position baseline

`classified_position` is the target and never a feature. Grid position and prior-history aggregates are available before the race.


In [ ]:
finish_data = frame[frame["classified_position"].notna() & frame["grid_position"].notna()]
finish_scores, finish_test_season = evaluate(finish_data, "classified_position")
finish_scores.round(3)


## Pit-stop-count baseline

This task starts in 2011, when recorded pit-event coverage begins. An absent pit row is treated as zero only within that supported era.


In [ ]:
pit_data = frame[frame["season"].ge(2011) & frame["pit_stop_count"].notna()]
pit_scores, pit_test_season = evaluate(pit_data, "pit_stop_count")
scores = pd.concat([finish_scores, pit_scores], ignore_index=True)
display(scores.round(3))
sns.barplot(data=scores, x="target", y="MAE", hue="model")
plt.title("Chronological holdout MAE (lower is better)")
plt.xlabel("")
plt.tight_layout()


## Responsible interpretation

- The latest season may be incomplete, so metrics will change after each race.
- Historical averages are shifted, but a stronger production pipeline should compute features race-by-race to handle duplicate historical entries explicitly.
- Race-start weather is excluded here to keep this a strict pre-event baseline.
- Use ranking metrics and uncertainty estimates before deploying finish predictions.
- Never replace the chronological holdout with a random row split.
